<a href="https://colab.research.google.com/github/xwang335/Campbell-A/blob/main/data_preprocess_corrected.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Preprocessing — Corrected Version

**Based on:** Gu, Kelly & Xiu (2020) "Empirical Asset Pricing via Machine Learning" (SSRN 3159577)

This notebook constructs the preprocessed dataset following the paper's methodology (Section 3.1):

1. Load 94 stock-level characteristics from `datashare.csv` (Green et al. 2017)
2. Query WRDS/CRSP for monthly returns and Fama-French risk-free rate
3. Construct excess returns and forward target variable `exret_lead1`
4. Load 8 macroeconomic predictors from Welch & Goyal (2008), lagged by 1 month
5. Cross-sectionally rank ALL 94 characteristics period-by-period → map to [-1, 1]
6. Filter sample to March 1957 – December 2016
7. Merge with macro data and save to Google Drive

**Key fixes vs. original `data_preprocess_` notebook:**
- Rank normalization applied to ALL 94 characteristics (paper footnote 29)
- Target variable `exret_lead1` created (forward excess return)
- Date range filtered to 1957-03 ~ 2016-12 (paper Section 3.1)
- Macro data loaded via gspread export (matching ols_3 pipeline, avoids NaN issue)
- Column drop properly assigned back
- Macro `dropna()` scoped to macro columns only (not all 57 columns)

## 0. Setup & Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q wrds gspread google-api-python-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 95.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 71.9 MB/s eta 0:00:00


In [3]:
import wrds
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('All imports OK')

All imports OK


## 1. Load datashare.csv (94 Stock-Level Characteristics)

The 94 predictive characteristics are based on Green et al. (2017), adapted from Jeremiah Green's SAS code.
The data is extended back to 1957. See paper Table A.6 for the full list.

In [4]:
csv_path = "/content/drive/MyDrive/datashare.csv"
df = pd.read_csv(csv_path)
df['DATE'] = pd.to_datetime(df['DATE'], format='%Y%m%d')

print(f"datashare shape: {df.shape}")
print(f"Date range: {df['DATE'].min().date()} to {df['DATE'].max().date()}")
print(f"Columns: {list(df.columns[:5])} ... {list(df.columns[-3:])}")
print(f"Number of characteristics: {df.shape[1] - 2}  (excluding permno, DATE)")

datashare shape: (4117300, 97)
Date range: 1957-01-31 to 2021-12-31
Columns: ['permno', 'DATE', 'mvel1', 'beta', 'betasq'] ... ['std_turn', 'zerotrade', 'sic2']
Number of characteristics: 95  (excluding permno, DATE)


## 2. Query WRDS for Monthly Returns (CRSP) and Risk-Free Rate (FF)

We obtain monthly total individual equity returns from CRSP for all firms listed in NYSE, AMEX, and NASDAQ.
Treasury-bill rate from Fama-French factors serves as the risk-free rate proxy.
Individual excess returns: `exret = ret - rf`

In [5]:
db = wrds.Connection()

Enter your WRDS username [root]:zixian_zhou
Enter your password:··········
WRDS recommends setting up a .pgpass file.
Create .pgpass file now [y/n]?: n
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [6]:
# Get date range from datashare to query WRDS
dates = df['DATE'].unique()
start_date = min(dates)
end_date = max(dates)

query = f"""
SELECT
    b.permno,
    b.date,
    b.ret,
    c.rf,
    (b.ret - c.rf) AS exret
FROM crsp.msf b
LEFT JOIN ff.factors_monthly c ON
    EXTRACT(YEAR FROM b.date) = EXTRACT(YEAR FROM c.date)
    AND EXTRACT(MONTH FROM b.date) = EXTRACT(MONTH FROM c.date)
WHERE b.date >= '{start_date}' AND b.date <= '{end_date}'
  AND b.ret IS NOT NULL
"""

df_msf_all = db.raw_sql(query)
df_msf_all['date'] = pd.to_datetime(df_msf_all['date'])

print(f"CRSP query returned: {len(df_msf_all):,} rows")
print(f"Date range: {df_msf_all['date'].min().date()} to {df_msf_all['date'].max().date()}")

CRSP query returned: 4,318,501 rows
Date range: 1957-01-31 to 2021-12-31


In [7]:
# Merge datashare characteristics with CRSP returns
df_exret = pd.merge(
    df,
    df_msf_all,
    left_on=['permno', 'DATE'],
    right_on=['permno', 'date'],
    how='inner'
)

# Drop redundant 'date' column (keep 'DATE')
df_exret = df_exret.drop(columns=['date'])

print(f"After merge: {len(df_exret):,} rows")

After merge: 4,096,791 rows


## 3. Basic Filters

- Keep rows where `ret` is not NaN (valid return)
- Keep rows where `mvel1` (market equity) is not NaN (needed for Top/Bottom 1000 analysis)

In [8]:
n_before = len(df_exret)
df_exret = df_exret[df_exret['ret'].notna()].copy()
df_exret = df_exret[df_exret['mvel1'].notna()].copy()
df_exret = df_exret.reset_index(drop=True)

print(f"Rows dropped by ret/mvel1 filter: {n_before - len(df_exret):,}")
print(f"Remaining rows: {len(df_exret):,}")

Rows dropped by ret/mvel1 filter: 2,982
Remaining rows: 4,093,809


## 4. Create Target Variable: `exret_lead1`

The prediction target is the **next month's excess return** for each stock:

$$r_{i,t+1} = \text{ret}_{i,t+1} - r_{f,t+1}$$

We create this by shifting `exret` backward by 1 within each `permno` group.
This must be done **before** date filtering to avoid losing the last month's target.

In [9]:
# Sort by permno and DATE to ensure correct shift
df_exret = df_exret.sort_values(['permno', 'DATE']).reset_index(drop=True)

# Create forward excess return target
df_exret['exret_lead1'] = df_exret.groupby('permno')['exret'].shift(-1)

print(f"exret_lead1 created. NaN count: {df_exret['exret_lead1'].isna().sum():,}")
print(f"(NaN expected for the last observation of each permno)")

exret_lead1 created. NaN count: 32,750
(NaN expected for the last observation of each permno)


## 5. Filter to Paper's Sample Window

Paper Section 3.1: "Our sample begins in March 1957 (the start date of the S&P 500) and ends in December 2016, totaling 60 years."

Also drop rows where `exret_lead1` is NaN (last obs per stock, or gaps).

In [10]:
n_before = len(df_exret)

# Filter to paper's sample window: 1957-03 to 2016-12
df_exret = df_exret[
    (df_exret['DATE'] >= pd.Timestamp('1957-03-31')) &
    (df_exret['DATE'] <= pd.Timestamp('2016-12-31'))
].copy()

# Drop rows without valid target
df_exret = df_exret.dropna(subset=['exret_lead1']).reset_index(drop=True)

print(f"Rows dropped by date/target filter: {n_before - len(df_exret):,}")
print(f"Final sample: {len(df_exret):,} rows")
print(f"Date range: {df_exret['DATE'].min().date()} to {df_exret['DATE'].max().date()}")

Rows dropped by date/target filter: 381,001
Final sample: 3,712,808 rows
Date range: 1957-04-30 to 2016-12-30


## 6. Cross-Sectional Rank Normalization of ALL 94 Characteristics

Paper footnote 29: *"We cross-sectionally rank all stock characteristics period-by-period and map these ranks into the [-1,1] interval."*

Paper footnote 30: *"Another issue is missing characteristics, which we replace with the cross-sectional median at each month for each stock, respectively."*

**Procedure (per month, per characteristic):**
1. Fill missing values with the cross-sectional median of that month
2. Rank all stocks (ties averaged)
3. Map ranks to [-1, 1]: `mapped = (rank / (N+1)) * 2 - 1`

In [11]:
# Identify the 94 characteristic columns
# In datashare.csv, columns 0='permno', 1='DATE', then 2:96 are the 94 characteristics
# After merge with CRSP, extra columns (ret, rf, exret, exret_lead1) are appended
NON_CHAR_COLS = ['permno', 'DATE', 'ret', 'rf', 'exret', 'exret_lead1']
CHAR_COLS_94 = [c for c in df_exret.columns if c not in NON_CHAR_COLS]

print(f"Number of characteristic columns to normalize: {len(CHAR_COLS_94)}")
print(f"First 10: {CHAR_COLS_94[:10]}")
print(f"Last 5:  {CHAR_COLS_94[-5:]}")

Number of characteristic columns to normalize: 95
First 10: ['mvel1', 'beta', 'betasq', 'chmom', 'dolvol', 'idiovol', 'indmom', 'mom1m', 'mom6m', 'mom12m']
Last 5:  ['retvol', 'std_dolvol', 'std_turn', 'zerotrade', 'sic2']


In [12]:
def rank_norm_median(s: pd.Series) -> pd.Series:
    """
    Cross-sectional rank normalization for one month:
    1) Fill NaN with cross-sectional median
    2) Rank (average ties)
    3) Map to [-1, 1]
    """
    med = s.median(skipna=True)
    filled = s.fillna(med)
    # If ALL values are NaN (median is NaN), fill with 0
    filled = filled.fillna(0)
    ranks = filled.rank(method='average')
    n = ranks.count()
    if n == 0:
        return filled  # edge case
    return (ranks / (n + 1)) * 2 - 1

# Apply rank normalization to all 94 characteristics, grouped by DATE (month)
print("Rank-normalizing all 94 characteristics cross-sectionally...")
df_exret[CHAR_COLS_94] = df_exret.groupby('DATE')[CHAR_COLS_94].transform(rank_norm_median)
print("Done!")

# Quick sanity check
print(f"\nSanity check — mvel1 stats:")
print(df_exret['mvel1'].describe())

Rank-normalizing all 94 characteristics cross-sectionally...
Done!

Sanity check — mvel1 stats:
count    3.712808e+06
mean     2.151067e-17
std      5.772389e-01
min     -9.997770e-01
25%     -4.999109e-01
50%      0.000000e+00
75%      4.999099e-01
max      9.997770e-01
Name: mvel1, dtype: float64


## 7. Load Macroeconomic Predictors

Paper Section 3.1: *"We also construct eight macroeconomic predictors following the variable definitions detailed in Welch and Goyal (2008), including dividend-price ratio (dp), earnings-price ratio (ep), book-to-market ratio (bm), net equity expansion (ntis), Treasury-bill rate (tbl), term spread (tms), default spread (dfy), and stock variance (svar)."*

Macro variables are **lagged by 1 month** to avoid look-ahead bias.

**Method:** Export Google Sheet "Data2024" to xlsx via gspread API, then read the "Monthly" sheet.
(This matches the ols_3 pipeline and avoids the NaN issue from direct pd.read_excel on a Google Sheet file.)

In [13]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
from googleapiclient.discovery import build

# Authorize
creds, _ = default()
gc = gspread.authorize(creds)

# Open Google Sheet "Data2024"
SHEET_NAME = "Data2024"
sh = gc.open(SHEET_NAME)

# Export to xlsx using Drive API
drive_service = build('drive', 'v3', credentials=creds)
file_id = sh.id
export_path = "/content/Data2024_export.xlsx"

request = drive_service.files().export_media(
    fileId=file_id,
    mimeType='application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'
)

with open(export_path, "wb") as f:
    f.write(request.execute())

print(f"Exported Google Sheet to: {export_path}")

# Read the Monthly sheet
df_macro = pd.read_excel(export_path, sheet_name="Monthly")
print(f"df_macro shape: {df_macro.shape}")
print(f"yyyymm range: {df_macro['yyyymm'].min()} to {df_macro['yyyymm'].max()}")

Exported Google Sheet to: /content/Data2024_export.xlsx
df_macro shape: (1848, 57)
yyyymm range: 187101 to 202412


In [14]:
# The 8 macro variables from Welch & Goyal (2008)
MACRO_COLS = ['tbl', 'd/p', 'e/p', 'b/m', 'tms', 'dfy', 'ntis', 'svar']

# Sort by date and lag by 1 month (shift forward) to avoid look-ahead bias
df_macro = df_macro.sort_values('yyyymm').reset_index(drop=True)
df_macro[MACRO_COLS] = df_macro[MACRO_COLS].shift(1)

# Create date columns for merge
df_macro['date'] = pd.to_datetime(df_macro['yyyymm'], format='%Y%m')
df_macro['year_month'] = df_macro['date'].dt.to_period('M')

# Select only needed columns (NOT dropna on all 57 columns — that was the original bug)
df_macro = df_macro[['yyyymm'] + MACRO_COLS + ['year_month']]

print(f"Macro data after lag: {len(df_macro)} rows")
print(f"NaN in macro cols after shift(1):")
print(df_macro[MACRO_COLS].isna().sum())

Macro data after lag: 1848 rows
NaN in macro cols after shift(1):
tbl     589
d/p       1
e/p       1
b/m     603
tms     589
dfy     577
ntis    672
svar    170
dtype: int64


## 8. Merge Stock Data with Macro Data

Merge on `year_month` (Period) with a left join — stock data is the primary table.

In [15]:
# Create year_month merge key for stock data
df_exret['year_month'] = df_exret['DATE'].dt.to_period('M')

# Left merge
df_merged = pd.merge(df_exret, df_macro, on='year_month', how='left')

# Drop helper columns (ASSIGN BACK — this was a bug in original preprocess)
df_merged = df_merged.drop(columns=['yyyymm', 'year_month'])

print(f"Merged shape: {df_merged.shape}")
print(f"\nMacro NaN check (should be minimal within 1957-2016):")
print(df_merged[MACRO_COLS].isna().sum())

Merged shape: (3712808, 109)

Macro NaN check (should be minimal within 1957-2016):
tbl     0
d/p     0
e/p     0
b/m     0
tms     0
dfy     0
ntis    0
svar    0
dtype: int64


## 9. Forward-fill SIC2 (Industry Code)

74 industry dummies are constructed from the first two digits of Standard Industrial Classification (SIC) codes.
Forward-fill (and backward-fill) within each stock to handle gaps.

In [16]:
if 'sic2' in df_merged.columns:
    n_nan_before = df_merged['sic2'].isna().sum()
    df_merged['sic2'] = df_merged.groupby('permno')['sic2'].ffill().bfill()
    n_nan_after = df_merged['sic2'].isna().sum()
    print(f"sic2 NaN: {n_nan_before:,} → {n_nan_after:,}")
else:
    print("Warning: 'sic2' column not found in data")

sic2 NaN: 0 → 0


## 10. Final Validation & Summary

In [17]:
print("=" * 70)
print("FINAL PREPROCESSED DATA SUMMARY")
print("=" * 70)
print(f"Shape:           {df_merged.shape}")
print(f"Date range:      {df_merged['DATE'].min().date()} to {df_merged['DATE'].max().date()}")
print(f"Unique permnos:  {df_merged['permno'].nunique():,}")
print(f"Unique months:   {df_merged['DATE'].nunique()}")
print(f"\nTarget variable (exret_lead1):")
print(f"  NaN count:     {df_merged['exret_lead1'].isna().sum()}")
print(f"  Mean:          {df_merged['exret_lead1'].mean():.6f}")
print(f"  Std:           {df_merged['exret_lead1'].std():.6f}")
print(f"\nRank-normalized characteristics (sample — mvel1, bm, mom12m):")
for c in ['mvel1', 'bm', 'mom12m']:
    print(f"  {c:10s}  min={df_merged[c].min():.4f}  max={df_merged[c].max():.4f}  mean={df_merged[c].mean():.6f}")
print(f"\nMacro variables NaN count:")
for c in MACRO_COLS:
    print(f"  {c:6s}  NaN: {df_merged[c].isna().sum():,}")
print(f"\nColumns ({len(df_merged.columns)}):")
print(list(df_merged.columns))

FINAL PREPROCESSED DATA SUMMARY
Shape:           (3712808, 109)
Date range:      1957-04-30 to 2016-12-30
Unique permnos:  29,825
Unique months:   717

Target variable (exret_lead1):
  NaN count:     0
  Mean:          0.007313
  Std:           0.172393

Rank-normalized characteristics (sample — mvel1, bm, mom12m):
  mvel1       min=-0.9998  max=0.9998  mean=0.000000
  bm          min=-0.9990  max=0.9990  mean=0.000000
  mom12m      min=-0.9955  max=0.9956  mean=0.000000

Macro variables NaN count:
  tbl     NaN: 0
  d/p     NaN: 0
  e/p     NaN: 0
  b/m     NaN: 0
  tms     NaN: 0
  dfy     NaN: 0
  ntis    NaN: 0
  svar    NaN: 0

Columns (109):
['permno', 'DATE', 'mvel1', 'beta', 'betasq', 'chmom', 'dolvol', 'idiovol', 'indmom', 'mom1m', 'mom6m', 'mom12m', 'mom36m', 'pricedelay', 'turn', 'absacc', 'acc', 'age', 'agr', 'bm', 'bm_ia', 'cashdebt', 'cashpr', 'cfp', 'cfp_ia', 'chatoia', 'chcsho', 'chempia', 'chinv', 'chpmia', 'convind', 'currat', 'depr', 'divi', 'divo', 'dy', 'egr', 'ep'

## 11. Close WRDS Connection & Save to Google Drive

In [18]:
db.close()
print("WRDS connection closed.")

WRDS connection closed.


In [19]:
import os

save_dir = '/content/drive/MyDrive/industry_project'
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, 'preprocess_data.csv')
df_merged.to_csv(save_path, index=False)

print(f"Saved to: {save_path}")
print(f"File size: {os.path.getsize(save_path) / 1e6:.1f} MB")

Saved to: /content/drive/MyDrive/industry_project/preprocess_data.csv
File size: 6274.2 MB


---

## Appendix: Discrepancies Fixed (Original `data_preprocess_` → This Version)

| Issue | Original `data_preprocess_` | This Version (Corrected) | Reference |
|-------|---------------------------|--------------------------|----------|
| **Rank normalization scope** | ALL 94 characteristics | ALL 94 characteristics | Paper fn.29: "rank all stock characteristics" |
| **Target variable** | NOT created | `exret_lead1 = shift(-1)` within permno | Paper Eq.1: predict $r_{i,t+1}$ |
| **Date filtering** | Only `>= 1957-03-01`, no end date | `1957-03-31 to 2016-12-31` | Paper Sec 3.1: "March 1957...December 2016" |
| **Macro data loading** | `pd.read_excel` on Drive path (may fail for Google Sheet) | gspread export → xlsx → read | Matches ols_3 pipeline |
| **Macro dropna** | `df_macro.dropna()` on all 57 columns (too aggressive) | No blanket dropna; NaN handled by left join | Prevents data loss |
| **Column drop** | `df_merged.drop(...)` NOT assigned back | `df_merged = df_merged.drop(...)` | Bug fix |
| **exret_lead1 NaN rows** | N/A (no target) | Dropped via `dropna(subset=['exret_lead1'])` | Consistent with ols_3 |

In [20]:
import os
import pandas as pd

# 原始 CSV 路径
csv_path = '/content/drive/MyDrive/industry_project/preprocess_data.csv'

# 输出 parquet 路径
parquet_path = '/content/drive/MyDrive/industry_project/preprocess_data.parquet'

# 读取 CSV
print("Loading CSV...")
df = pd.read_csv(csv_path)

print("Loaded.")
print(df.shape)

# 保存为 parquet
print("Saving to Parquet...")
df.to_parquet(parquet_path, index=False, engine='pyarrow')

print(f"Saved to: {parquet_path}")
print(f"Parquet size: {os.path.getsize(parquet_path) / 1e9:.2f} GB")

Loading CSV...
Loaded.
(3712808, 109)
Saving to Parquet...
Saved to: /content/drive/MyDrive/industry_project/preprocess_data.parquet
Parquet size: 1.93 GB
